In [ ]:
## login wandb
import wandb
wandb.login()
## set up project name
import os
os.environ["WANDB_PROJECT"] = "chess-llm" 
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [ ]:
import unsloth
import vllm
import torch
import trl

print(vllm.__version__)
print(unsloth.__version__)
print(torch.__version__)
print(trl.__version__)

## Model

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 1024 # Can increase for longer reasoning traces
lora_rank = 16 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-7B-Instruct", 
    max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    fast_inference = False, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9, # Reduce if out of memory
)

In [ ]:
NEW_TOKENS = [  
    "♔","♕","♖","♗","♘","♙",  
    "♚","♛","♜","♝","♞","♟",
    "<uci_move>", "</uci_move>",
]
xs = tokenizer("♔♕♖♗♘♙♚♛♜♝♞♟ <uci_move>a1a2</uci_move>")
print([tokenizer.decode(x) for x in xs["input_ids"]])

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 3407,
)

## Data

In [ ]:
from datasets import load_dataset, Dataset
from tqdm import tqdm
import chess
import pandas as pd

In [ ]:
SYSTEM_PROMPT = """
You are a chess expert.

Task:  
- Analyze the position.  
- Briefly explain the reasoning.  
- Choose the single best move and put inside the tags <uc_move>YOUR MOVE</uci_move>.

Rules:  
- The move MUST be from the legal moves list.   
- Max 120 words total.  

## Context
Your side: {side_to_move}
Legal moves: {legal_moves_uci_list}
Board position:
{board_utf}
"""

In [ ]:
def format_prompt(row):  
    prompt = SYSTEM_PROMPT.format(
        side_to_move=row["side_to_move"],
        legal_moves_uci_list=" ".join(row["legal_moves_uci_list"]),
        board_utf=row["board_utf"],
    ) 
    response = f"{row["explanation"]}\n<uci_move>{row["target_move"]}</uci_move>"   
    return [  
        {"role": "user", "content": prompt},  
        {"role": "assistant", "content": response},  
    ]

In [ ]:
from datasets import load_dataset  
  
dataset = load_dataset("Norrawee/chess-exp04")
df = dataset["train"].to_pandas()
df = df[:100]

In [ ]:
## preprocess
df["explanation"] = df["explanation"].apply(lambda x: x.replace("\n\n", "\n"))
df["prompt"] = df.apply(format_prompt, axis=1)
df["text"] = tokenizer.apply_chat_template(df["prompt"].values.tolist(), tokenize=False)

In [ ]:
from sklearn.model_selection import train_test_split  
from datasets import Dataset  
  
# Unique boards  
unique_boards = df["board_utf"].unique()  
  
# Split boards, NOT rows  
train_boards, test_boards = train_test_split(  
    unique_boards,  
    test_size=0.1,  
    random_state=42,  
    shuffle=True,  
)  
  
# Filter rows  
train_df = df[df["board_utf"].isin(train_boards)].reset_index(drop=True)  
test_df  = df[df["board_utf"].isin(test_boards)].reset_index(drop=True)  
  
# Create HF datasets  
ds = {  
    "train": Dataset.from_pandas(train_df),  
    "test": Dataset.from_pandas(test_df),  
}  

In [ ]:
text = ds["train"]["text"][0]

print(text)
print(len(tokenizer(text)["input_ids"]))

## SFT

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset= ds["train"],
    eval_dataset= ds["test"],
    args = SFTConfig(
        dataset_text_field = "text",
        optim = "adamw_8bit",
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "wandb", # Use TrackIO/WandB etc
        
        # training params
        learning_rate=5e-5,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=1,
        # num_train_epochs=5,
        fp16=False,
        bf16=True,
        weight_decay = 0.001,
        
        # logging
        # eval_strategy="epoch",
        # save_strategy="epoch",
        # logging_strategy="epoch",
        eval_strategy="steps",
        save_strategy="steps",
        logging_strategy="steps",
        logging_steps=10,
        save_steps=10,
        eval_steps=10,
        save_total_limit=1,
        max_steps=30,
    ),
)

In [ ]:
trainer.train()

## Test

In [ ]:
import re
UCI_PATTERN = re.compile(r"<uci_move>(.*?)</uci_move>")  
  
def extract_uci(text):  
    match = UCI_PATTERN.search(text)  
    return match.group(1).strip() if match else None  

In [ ]:
import chess  
import chess.engine  
from multiprocessing import cpu_count  
  
# ---------------- CONFIG ----------------  
ENGINE_PATH = "stockfish"   # change if needed  
ENGINE_LIMIT = chess.engine.Limit(depth=16)  
N_WORKERS = max(1, cpu_count() - 1)  
  
# ------------- WORKER STATE -------------  
engine = chess.engine.SimpleEngine.popen_uci(ENGINE_PATH)
  
# ----------- EVAL HELPERS ---------------  
def eval_cp_v1_style(info):  
    score = info["score"].relative  
    cp = score.score(mate_score=1000)  
    if cp is None:  
        return 1000  
    return int(max(-1000, min(1000, cp)))  
  
  
def compute_cpl_v1_style(board, move):  
    best_info = engine.analyse(board, ENGINE_LIMIT)  
    eval_before = eval_cp_v1_style(best_info)  
  
    board.push(move)  
    after_info = engine.analyse(board, ENGINE_LIMIT)  
    eval_after = eval_cp_v1_style(after_info)  
    board.pop()  
  
    eval_after_mover_pov = -eval_after  
    return max(0, eval_before - eval_after_mover_pov)   

def compute_chess_score(uci_move, fen_board):
    board = chess.Board(fen_board)
    move = chess.Move.from_uci(uci_move)
    score = compute_cpl_v1_style(board, move)
    return score

In [ ]:
def generate_batch(model, prompts):  
    texts = [  
        tokenizer.apply_chat_template(p, tokenize=False, add_generation_prompt=True)  
        for p in prompts  
    ]  
  
    model_inputs = tokenizer(  
        texts,  
        return_tensors="pt",  
        padding=True,  
    ).to(model.device)  
  
    generated = model.generate(  
        **model_inputs,  
        max_new_tokens=200,
        do_sample=True,
        temperature=0.01,
        top_p=0.8,
        top_k=20,
    )   
  
    decoded = tokenizer.batch_decode(  
        generated,  
        skip_special_tokens=True  
    )  
  
    return [t.split("assistant")[-1].strip() for t in decoded]  

In [ ]:
from tqdm import tqdm  
  
BATCH_SIZE = 1 ## padding affects the outputs (i dont know how to fix).
  
responses = []
for batch_start in tqdm(range(0, len(ds["test"]), BATCH_SIZE)):  
    batch = ds["test"][batch_start: batch_start + BATCH_SIZE]  
  
    prompts = [ex[:1] for ex in batch["prompt"]]  
    responses.extend(generate_batch(model, prompts))

In [ ]:
outputs = []

for i, (response, example) in enumerate(zip(responses, ds["test"])):
    pred_move = extract_uci(response)  
    legal_moves_uci_list = example["legal_moves_uci_list"]
    fen = example["board_fen"]

    if pred_move not in legal_moves_uci_list:
        legal = False
        score = -1000
    else:
        legal = True
        score = compute_chess_score(pred_move, fen)
    outputs.append({
        "score": score,
        "response": response,
        "legal": legal
    })
    
    if i < 10:
        print(example["board_utf"])
        print(response)
        print(score)
        print(f"Target move: {example["target_move"]}")
        print("*"*60)


In [ ]:
output_df = pd.DataFrame(outputs)
output_df[["legal"]].value_counts()

In [ ]:
output_df[["score"]].describe()

## Save

In [ ]:
model.push_to_hub_merged(
    "Norrawee/Qwen/Qwen2.5-7B-Instruct-sft-exp04", 
    tokenizer,
    save_method = "merged_16bit", 
)

In [ ]:
wandb.finish()